# ARC-AGI-3 LLM Agent

LLM-based agent using Qwen3.6-27B-FP8 via vLLM. 
Architecture: tool-calling REPL with Python sandbox, anti-loop detection, 
context management. Based on the Duck harness (MIT, Tufa Labs).

**Datasets required:**
- `driessmit1/arc3-vllm-h100-wheelhouse-v3` (vLLM + torch wheels)
- `driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot` (Qwen3.6-27B-FP8 model)


In [ ]:
import base64
import json
import os
import shutil
import subprocess
import sys
import sysconfig
import time
from pathlib import Path

# Competition rerun detection
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true", "yes"}
NOTEBOOK_START = time.time()

os.environ["MPLBACKEND"] = "Agg"

# CUDA library path
cuda_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    e for e in [cuda_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if e
)

WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)

print(f"TRUE_SUBMISSION={TRUE_SUBMISSION}")
print(f"GPU check:")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=False)


In [ ]:
# Install ARC runtime from competition wheels
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "--no-index", "--no-warn-conflicts", "--disable-pip-version-check",
    "--find-links", "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
    "arc-agi", "python-dotenv",
], stdout=subprocess.DEVNULL)
print("ARC runtime installed")


In [ ]:
# Install vLLM from the wheelhouse dataset
WHEELHOUSE_DATASET = "driessmit1/arc3-vllm-h100-wheelhouse-v3"
MODEL_DATASET = "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"

# Find the wheelhouse and model paths
def find_dataset_path(slug):
    for candidate in [
        Path(f"/kaggle/input/{slug}"),
        Path(f"/kaggle/input/datasets") / slug.split("/")[0] / slug.split("/")[1],
    ]:
        if candidate.exists():
            return candidate
    # Fallback: search for the slug directory
    for p in Path("/kaggle/input").iterdir():
        if slug.split("/")[-1] in p.name or p.name == slug.split("/")[-1]:
            return p
    return Path(f"/kaggle/input/{slug.split('/')[-1]}")

WHEELHOUSE = find_dataset_path(WHEELHOUSE_DATASET)
MODEL_PATH = find_dataset_path(MODEL_DATASET)

print(f"Wheelhouse: {WHEELHOUSE} (exists: {WHEELHOUSE.exists()})")
print(f"Model: {MODEL_PATH} (exists: {MODEL_PATH.exists()})")

# Install vLLM into a separate site-packages to avoid conflicts
SITE_PACKAGES = WORKING / "vllm_site"
SITE_PACKAGES.mkdir(exist_ok=True)

requirements = WHEELHOUSE / "requirements.lock"
if not requirements.exists():
    # Try alternative names
    for name in ["requirements.txt", "requirements.lock"]:
        alt = WHEELHOUSE / name
        if alt.exists():
            requirements = alt
            break

if requirements.exists():
    print(f"Installing vLLM from {requirements}")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--no-index", "--find-links", str(WHEELHOUSE),
        "--requirement", str(requirements),
        "--target", str(SITE_PACKAGES),
        "--upgrade", "--ignore-installed",
        "--only-binary", ":all:", "--no-compile",
        "--disable-pip-version-check", "--no-warn-conflicts",
    ])
    print("vLLM installed")
else:
    print(f"WARNING: No requirements file found in {WHEELHOUSE}")
    print(f"Contents: {list(WHEELHOUSE.iterdir())[:10]}")
    # Try direct install of vllm wheel
    wheels = list(WHEELHOUSE.glob("vllm-*.whl"))
    if wheels:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install",
            "--no-index", "--find-links", str(WHEELHOUSE),
            "--target", str(SITE_PACKAGES),
            str(wheels[0]),
        ])
        print(f"Installed vLLM wheel: {wheels[0].name}")

# Add to Python path
os.environ["PYTHONPATH"] = str(SITE_PACKAGES) + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, str(SITE_PACKAGES))


In [ ]:
import urllib.request

VLLM_HOST = "127.0.0.1"
VLLM_PORT = 1234
VLLM_BASE_URL = f"http://{VLLM_HOST}:{VLLM_PORT}/v1"
SERVED_MODEL = "vrfai/Qwen3.6-27B-FP8"
VLLM_LOG = WORKING / "vllm_server.log"
VLLM_PID_FILE = WORKING / "vllm_server.pid"

# Kill any existing server
if VLLM_PID_FILE.exists():
    try:
        os.kill(int(VLLM_PID_FILE.read_text().strip()), 9)
    except Exception:
        pass
    VLLM_PID_FILE.unlink(missing_ok=True)

# Start vLLM OpenAI-compatible server
vllm_env = os.environ.copy()
vllm_env.update({
    "USE_TF": "0",
    "TRANSFORMERS_NO_TF": "1",
    "TRANSFORMERS_NO_TORCHVISION": "1",
    "VLLM_NO_USAGE_STATS": "1",
    "PYTHONPATH": str(SITE_PACKAGES),
})

cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", str(MODEL_PATH),
    "--served-model-name", SERVED_MODEL,
    "--host", VLLM_HOST,
    "--port", str(VLLM_PORT),
    "--tensor-parallel-size", "1",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "qwen3_coder",
    "--generation-config", "vllm",
    "--enable-prefix-caching",
    "--default-chat-template-kwargs", '{"preserve_thinking": true}',
    "--reasoning-parser", "qwen3",
    "--max-model-len", "65536",
]

print(f"Starting vLLM: {' '.join(cmd[:8])}...")
log_fh = open(VLLM_LOG, "w")
proc = subprocess.Popen(cmd, env=vllm_env, stdout=log_fh, stderr=subprocess.STDOUT, text=True)
VLLM_PID_FILE.write_text(str(proc.pid))

# Wait for server to be ready
print("Waiting for vLLM server...", flush=True)
deadline = time.time() + 600
ready = False
while time.time() < deadline:
    try:
        req = urllib.request.Request(f"{VLLM_BASE_URL}/models")
        with urllib.request.urlopen(req, timeout=5) as resp:
            if resp.status < 500:
                ready = True
                break
    except Exception:
        pass
    time.sleep(5)
    # Check if process is still alive
    if proc.poll() is not None:
        print(f"vLLM server exited with code {proc.returncode}")
        print(f"Log tail: {VLLM_LOG.read_text()[-2000:]}")
        break

if ready:
    print("vLLM server ready!")
    # Smoke test
    payload = json.dumps({
        "model": SERVED_MODEL,
        "messages": [{"role": "user", "content": "Say hello in one word."}],
        "max_tokens": 16,
        "temperature": 0.0,
    }).encode()
    req = urllib.request.Request(
        f"{VLLM_BASE_URL}/chat/completions",
        data=payload,
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        result = json.loads(resp.read())
    reply = result["choices"][0]["message"].get("content", "").strip()
    print(f"Smoke test reply: {reply!r}")
    VLLM_OK = True
else:
    print("ERROR: vLLM server did not start. Log:")
    print(VLLM_LOG.read_text()[-3000:])
    VLLM_OK = False


In [ ]:

# Deploy agent source files
agent_dir = Path("/tmp/arc_llm_agent")
agent_dir.mkdir(parents=True, exist_ok=True)

(agent_dir / "__init__.py").write_text("""# LLM agent package""")
(agent_dir / "grid_utils.py").write_bytes(base64.b64decode("IiIiR3JpZCBmb3JtYXR0aW5nIGFuZCBzZWdtZW50YXRpb24gdXRpbGl0aWVzIGZvciBBUkMtQUdJLTMgTExNIGFnZW50LgoKRGVyaXZlZCBmcm9tIHRoZSBEdWNrIGhhcm5lc3MgKE1JVCBsaWNlbnNlLCBUdWZhIExhYnMpLgpTaW1wbGlmaWVkIGZvciB1c2Ugd2l0aCB0aGUgQVJDLUFHSS0zLUFnZW50cyBmcmFtZXdvcmsuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgaGFzaGxpYgpmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBTZXF1ZW5jZQoKIyBBUkMgY29sb3IgcGFsZXR0ZTogaW5kZXggLT4gc2luZ2xlLWNoYXIgbGFiZWwKQVJDX0NPTE9SX0NIQVJTID0gIld3Z0djQk1QUmJTWU9yTnAiCkFSQ19DT0xPUl9MRUdFTkQgPSAoCiAgICAiVz13aGl0ZSwgdz1saWdodCBncmF5LCBnPWdyYXksIEc9ZGFyayBncmF5LCBjPWNoYXJjb2FsLCBCPWJsYWNrLCAiCiAgICAiTT1tYWdlbnRhLCBQPXBpbmssIFI9cmVkLCBiPWJsdWUsIFM9c2t5IGJsdWUsIFk9eWVsbG93LCBPPW9yYW5nZSwgIgogICAgInI9ZGFyayByZWQsIE49bGlnaHQgZ3JlZW4sIHA9cHVycGxlIgopCgojIEFSQyBjb2xvciBwYWxldHRlOiBpbmRleCAtPiBSR0IKQVJDX0NPTE9SX01BUDogZGljdFtpbnQsIHR1cGxlW2ludCwgaW50LCBpbnRdXSA9IHsKICAgIDA6ICgyNTUsIDI1NSwgMjU1KSwgMTogKDIwNCwgMjA0LCAyMDQpLCAyOiAoMTUzLCAxNTMsIDE1MyksCiAgICAzOiAoMTAyLCAxMDIsIDEwMiksIDQ6ICg1MSwgNTEsIDUxKSwgICAgNTogKDAsIDAsIDApLAogICAgNjogKDIyOSwgNTgsIDE2MyksICA3OiAoMjU1LCAxMjMsIDIwNCksIDg6ICgyNDksIDYwLCA0OSksCiAgICA5OiAoMzAsIDE0NywgMjU1KSwgIDEwOiAoMTM2LCAyMTYsIDI0MSksIDExOiAoMjU1LCAyMjAsIDApLAogICAgMTI6ICgyNTUsIDEzMywgMjcpLCAxMzogKDE0NiwgMTgsIDQ5KSwgIDE0OiAoNzksIDIwNCwgNDgpLAogICAgMTU6ICgxNjMsIDg2LCAyMTQpLAp9CgoKZGVmIGZvcm1hdF9ncmlkX2FzY2lpKGdyaWQ6IFNlcXVlbmNlW1NlcXVlbmNlW2ludF1dKSAtPiBzdHI6CiAgICAiIiJSZW5kZXIgYSBudW1lcmljIGdyaWQgYXMgQVNDSUkgYXJ0IHVzaW5nIEFSQyBjb2xvciBjaGFyYWN0ZXJzLiIiIgogICAgaWYgbm90IGdyaWQ6CiAgICAgICAgcmV0dXJuICIoZW1wdHkgZ3JpZCkiCiAgICBsaW5lcyA9IFtdCiAgICBmb3Igcm93IGluIGdyaWQ6CiAgICAgICAgY2hhcnMgPSBbXQogICAgICAgIGZvciB2IGluIHJvdzoKICAgICAgICAgICAgY2hhcnMuYXBwZW5kKEFSQ19DT0xPUl9DSEFSU1ttYXgoMCwgbWluKDE1LCBpbnQodikpKV0pCiAgICAgICAgbGluZXMuYXBwZW5kKCIiLmpvaW4oY2hhcnMpKQogICAgcmV0dXJuICJcbiIuam9pbihsaW5lcykKCgpkZWYgZ3JpZF90b19wbmdfYnl0ZXMoZ3JpZDogU2VxdWVuY2VbU2VxdWVuY2VbaW50XV0sIHVwc2NhbGU6IGludCA9IDQpIC0+IGJ5dGVzOgogICAgIiIiUmVuZGVyIGEgbnVtZXJpYyBncmlkIGFzIGEgUE5HIGltYWdlIChubyBQSUwgZGVwZW5kZW5jeSDigJQgcHVyZSBieXRlcykuIiIiCiAgICB0cnk6CiAgICAgICAgZnJvbSBQSUwgaW1wb3J0IEltYWdlCiAgICAgICAgaW1wb3J0IGlvCiAgICBleGNlcHQgSW1wb3J0RXJyb3I6CiAgICAgICAgcmV0dXJuIGIiIgogICAgCiAgICByb3dzID0gbGVuKGdyaWQpCiAgICBjb2xzID0gbWF4KChsZW4ocikgZm9yIHIgaW4gZ3JpZCksIGRlZmF1bHQ9MCkgaWYgcm93cyBlbHNlIDAKICAgIGlmIHJvd3MgPD0gMCBvciBjb2xzIDw9IDA6CiAgICAgICAgcmV0dXJuIGIiIgogICAgCiAgICBpbWFnZSA9IEltYWdlLm5ldygiUkdCIiwgKGNvbHMsIHJvd3MpLCBBUkNfQ09MT1JfTUFQWzBdKQogICAgcGl4ZWxzID0gaW1hZ2UubG9hZCgpCiAgICBmb3Igcm93X2lkeCwgcm93IGluIGVudW1lcmF0ZShncmlkKToKICAgICAgICBmb3IgY29sX2lkeCBpbiByYW5nZShjb2xzKToKICAgICAgICAgICAgdmFsdWUgPSByb3dbY29sX2lkeF0gaWYgY29sX2lkeCA8IGxlbihyb3cpIGVsc2UgMAogICAgICAgICAgICBwaXhlbHNbY29sX2lkeCwgcm93X2lkeF0gPSBBUkNfQ09MT1JfTUFQLmdldChpbnQodmFsdWUpLCBBUkNfQ09MT1JfTUFQWzBdKQogICAgCiAgICBpZiB1cHNjYWxlID4gMToKICAgICAgICBpbWFnZSA9IGltYWdlLnJlc2l6ZSgoY29scyAqIHVwc2NhbGUsIHJvd3MgKiB1cHNjYWxlKSwgSW1hZ2UuUmVzYW1wbGluZy5ORUFSRVNUKQogICAgCiAgICBidWZmZXIgPSBpby5CeXRlc0lPKCkKICAgIGltYWdlLnNhdmUoYnVmZmVyLCBmb3JtYXQ9IlBORyIpCiAgICByZXR1cm4gYnVmZmVyLmdldHZhbHVlKCkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIFNlZ21lbnRhdGlvbiAoY29ubmVjdGVkLWNvbXBvbmVudCBhbmFseXNpcykKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCl9PUlRIID0gKCgtMSwgMCksICgxLCAwKSwgKDAsIC0xKSwgKDAsIDEpKQpfQ1cgPSAoKC0xLCAtMSksICgtMSwgMCksICgtMSwgMSksICgwLCAxKSwgKDEsIDEpLCAoMSwgMCksICgxLCAtMSksICgwLCAtMSkpCl9DV19JTkRFWCA9IHtvZmY6IGkgZm9yIGksIG9mZiBpbiBlbnVtZXJhdGUoX0NXKX0KCgpkZWYgX3RyYWNlX291dGVyX2NvbnRvdXIoY2VsbHMsIHN0YXJ0KToKICAgICIiIk1vb3JlLW5laWdoYm91ciB0cmFjZSBvZiBvdXRlciBwZXJpbWV0ZXIsIGNsb2Nrd2lzZS4iIiIKICAgIGlmIGxlbihjZWxscykgPT0gMToKICAgICAgICByZXR1cm4gW3N0YXJ0XQogICAgY29udG91ciA9IFtzdGFydF0KICAgIGIgPSBzdGFydAogICAgcHJldiA9IChzdGFydFswXSwgc3RhcnRbMV0gLSAxKQogICAgc2Vjb25kID0gTm9uZQogICAgZm9yIF8gaW4gcmFuZ2UoOCAqIGxlbihjZWxscykgKyAxNik6CiAgICAgICAgaWR4ID0gX0NXX0lOREVYWyhwcmV2WzBdIC0gYlswXSwgcHJldlsxXSAtIGJbMV0pXQogICAgICAgIG54dCA9IE5vbmUKICAgICAgICBmb3IgayBpbiByYW5nZSgxLCA5KToKICAgICAgICAgICAgb2ZmID0gX0NXWyhpZHggKyBrKSAlIDhdCiAgICAgICAgICAgIGNhbmQgPSAoYlswXSArIG9mZlswXSwgYlsxXSArIG9mZlsxXSkKICAgICAgICAgICAgaWYgY2FuZCBpbiBjZWxsczoKICAgICAgICAgICAgICAgIG54dCA9IGNhbmQKICAgICAgICAgICAgICAgIGJhY2sgPSBfQ1dbKGlkeCArIGsgLSAxKSAlIDhdCiAgICAgICAgICAgICAgICBuZXdfcHJldiA9IChiWzBdICsgYmFja1swXSwgYlsxXSArIGJhY2tbMV0pCiAgICAgICAgICAgICAgICBicmVhawogICAgICAgIGlmIG54dCBpcyBOb25lOgogICAgICAgICAgICBicmVhawogICAgICAgIGlmIHNlY29uZCBpcyBOb25lOgogICAgICAgICAgICBzZWNvbmQgPSBueHQKICAgICAgICBlbGlmIGIgPT0gc3RhcnQgYW5kIG54dCA9PSBzZWNvbmQ6CiAgICAgICAgICAgIGJyZWFrCiAgICAgICAgY29udG91ci5hcHBlbmQobnh0KQogICAgICAgIHByZXYsIGIgPSBuZXdfcHJldiwgbnh0CiAgICBpZiBsZW4oY29udG91cikgPiAxIGFuZCBjb250b3VyWy0xXSA9PSBjb250b3VyWzBdOgogICAgICAgIGNvbnRvdXIucG9wKCkKICAgIHJldHVybiBjb250b3VyCgoKZGVmIF9jb3JuZXJfcG9pbnRzKGNvbnRvdXIpOgogICAgIiIiUmVkdWNlIGNvbnRvdXIgdG8gZGlyZWN0aW9uLWNoYW5nZSBwb2ludHMuIiIiCiAgICBpZiBsZW4oY29udG91cikgPD0gMjoKICAgICAgICByZXR1cm4gbGlzdChjb250b3VyKQogICAgbSA9IGxlbihjb250b3VyKQogICAgY29ybmVycyA9IFtdCiAgICBmb3IgaSBpbiByYW5nZShtKToKICAgICAgICBwcmV2X2MsIGN1ciwgbnh0ID0gY29udG91cltpIC0gMV0sIGNvbnRvdXJbaV0sIGNvbnRvdXJbKGkgKyAxKSAlIG1dCiAgICAgICAgZF9pbiA9IChjdXJbMF0gLSBwcmV2X2NbMF0sIGN1clsxXSAtIHByZXZfY1sxXSkKICAgICAgICBkX291dCA9IChueHRbMF0gLSBjdXJbMF0sIG54dFsxXSAtIGN1clsxXSkKICAgICAgICBpZiBkX2luICE9IGRfb3V0OgogICAgICAgICAgICBjb3JuZXJzLmFwcGVuZChjdXIpCiAgICByZXR1cm4gY29ybmVycwoKCmRlZiBfb2JqZWN0X2hhc2goY2VsbHMsIGNvbG9yKToKICAgICIiIlRyYW5zbGF0aW9uLWludmFyaWFudCBzaWduYXR1cmUgb2YgYW4gb2JqZWN0LiIiIgogICAgbWluX3IgPSBtaW4ociBmb3IgciwgXyBpbiBjZWxscykKICAgIG1pbl9jID0gbWluKGMgZm9yIF8sIGMgaW4gY2VsbHMpCiAgICBub3JtID0gc29ydGVkKChyIC0gbWluX3IsIGMgLSBtaW5fYykgZm9yIHIsIGMgaW4gY2VsbHMpCiAgICBwYXlsb2FkID0gcmVwcigoY29sb3IsIG5vcm0pKS5lbmNvZGUoKQogICAgcmV0dXJuIGhhc2hsaWIuc2hhMShwYXlsb2FkKS5oZXhkaWdlc3QoKVs6MTZdCgoKZGVmIHNlZ21lbnRfZ3JpZChncmlkOiBTZXF1ZW5jZVtTZXF1ZW5jZVtpbnRdXSkgLT4gZGljdFtzdHIsIEFueV06CiAgICAiIiJTZWdtZW50IGEgZ3JpZCBpbnRvIGNvbm5lY3RlZCBjb21wb25lbnRzLgogICAgCiAgICBSZXR1cm5zIGRpY3Qgd2l0aDoKICAgICAgLSBub2RlczogbGlzdCBvZiB7aWQsIGNvbG9yLCBoYXNoLCBwaXhlbHMsIGJvdW5kYXJ5LCBjaGlsZHJlbn0KICAgICAgLSBhZGphY2VuY3lfbGlzdDogbGlzdCBvZiBbaSwgal0gcGFpcnMKICAgICIiIgogICAgaGVpZ2h0ID0gbGVuKGdyaWQpCiAgICB3aWR0aCA9IGxlbihncmlkWzBdKSBpZiBoZWlnaHQgZWxzZSAwCiAgICBpZiBub3QgaGVpZ2h0IG9yIG5vdCB3aWR0aDoKICAgICAgICByZXR1cm4geyJub2RlcyI6IFtdLCAiYWRqYWNlbmN5X2xpc3QiOiBbXX0KICAgIAogICAgIyBDb25uZWN0ZWQgY29tcG9uZW50cyB2aWEgZmxvb2QgZmlsbAogICAgY29tcF9pZCA9IFtbLTFdICogd2lkdGggZm9yIF8gaW4gcmFuZ2UoaGVpZ2h0KV0KICAgIGNvbXBvbmVudHMgPSBbXQogICAgCiAgICBmb3Igc3IgaW4gcmFuZ2UoaGVpZ2h0KToKICAgICAgICBmb3Igc2MgaW4gcmFuZ2Uod2lkdGgpOgogICAgICAgICAgICBpZiBjb21wX2lkW3NyXVtzY10gIT0gLTE6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICB2YWx1ZSA9IGdyaWRbc3JdW3NjXQogICAgICAgICAgICBjaWQgPSBsZW4oY29tcG9uZW50cykKICAgICAgICAgICAgY2VsbHMgPSBzZXQoKQogICAgICAgICAgICBzdGFjayA9IFsoc3IsIHNjKV0KICAgICAgICAgICAgd2hpbGUgc3RhY2s6CiAgICAgICAgICAgICAgICByLCBjID0gc3RhY2sucG9wKCkKICAgICAgICAgICAgICAgIGlmIChyLCBjKSBpbiBjZWxsczoKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgaWYgMCA8PSByIDwgaGVpZ2h0IGFuZCAwIDw9IGMgPCB3aWR0aCBhbmQgZ3JpZFtyXVtjXSA9PSB2YWx1ZSBhbmQgY29tcF9pZFtyXVtjXSA9PSAtMToKICAgICAgICAgICAgICAgICAgICBjZWxscy5hZGQoKHIsIGMpKQogICAgICAgICAgICAgICAgICAgIGNvbXBfaWRbcl1bY10gPSBjaWQKICAgICAgICAgICAgICAgICAgICBmb3IgZHIsIGRjIGluIF9PUlRIOgogICAgICAgICAgICAgICAgICAgICAgICBuciwgbmMgPSByICsgZHIsIGMgKyBkYwogICAgICAgICAgICAgICAgICAgICAgICBpZiAwIDw9IG5yIDwgaGVpZ2h0IGFuZCAwIDw9IG5jIDwgd2lkdGggYW5kIGNvbXBfaWRbbnJdW25jXSA9PSAtMSBhbmQgZ3JpZFtucl1bbmNdID09IHZhbHVlOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgc3RhY2suYXBwZW5kKChuciwgbmMpKQogICAgICAgICAgICBjb21wb25lbnRzLmFwcGVuZCh7InZhbHVlIjogdmFsdWUsICJjZWxscyI6IGNlbGxzLCAic3RhcnQiOiAoc3IsIHNjKX0pCiAgICAKICAgICMgQnVpbGQgbm9kZXMKICAgIG5vZGVzID0gW10KICAgIGZvciBjaWQsIGNvbXAgaW4gZW51bWVyYXRlKGNvbXBvbmVudHMpOgogICAgICAgIGNvbG9yX2NoYXIgPSBBUkNfQ09MT1JfQ0hBUlNbbWF4KDAsIG1pbigxNSwgY29tcFsidmFsdWUiXSkpXQogICAgICAgIG9ial9oYXNoID0gX29iamVjdF9oYXNoKGNvbXBbImNlbGxzIl0sIGNvbXBbInZhbHVlIl0pCiAgICAgICAgY29udG91ciA9IF90cmFjZV9vdXRlcl9jb250b3VyKGNvbXBbImNlbGxzIl0sIGNvbXBbInN0YXJ0Il0pCiAgICAgICAgYm91bmRhcnkgPSBfY29ybmVyX3BvaW50cyhjb250b3VyKQogICAgICAgIG5vZGVzLmFwcGVuZCh7CiAgICAgICAgICAgICJpZCI6IGNpZCwKICAgICAgICAgICAgImNvbG9yIjogY29sb3JfY2hhciwKICAgICAgICAgICAgImhhc2giOiBvYmpfaGFzaCwKICAgICAgICAgICAgInBpeGVscyI6IGxlbihjb21wWyJjZWxscyJdKSwKICAgICAgICAgICAgImJvdW5kYXJ5IjogW1tyLCBjXSBmb3IgciwgYyBpbiBib3VuZGFyeVs6MTJdXSwgICMgbGltaXQgc2l6ZQogICAgICAgICAgICAiY2hpbGRyZW4iOiBbXSwKICAgICAgICB9KQogICAgCiAgICAjIENvbXB1dGUgY29udGFpbm1lbnQgKGNoaWxkcmVuKQogICAgZm9yIG91dGVyX2lkLCBvdXRlcl9jb21wIGluIGVudW1lcmF0ZShjb21wb25lbnRzKToKICAgICAgICBvdXRlcl9jZWxscyA9IG91dGVyX2NvbXBbImNlbGxzIl0KICAgICAgICBmb3IgaW5uZXJfaWQsIGlubmVyX2NvbXAgaW4gZW51bWVyYXRlKGNvbXBvbmVudHMpOgogICAgICAgICAgICBpZiBpbm5lcl9pZCA9PSBvdXRlcl9pZDoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICMgQ2hlY2sgaWYgaW5uZXIgaXMgZnVsbHkgZW5jbG9zZWQgYnkgb3V0ZXIKICAgICAgICAgICAgaW5uZXJfY2VsbHMgPSBpbm5lcl9jb21wWyJjZWxscyJdCiAgICAgICAgICAgIGlmIGFsbChhbnkoKHIgKyBkciwgYyArIGRjKSBpbiBvdXRlcl9jZWxscyBmb3IgZHIsIGRjIGluIF9PUlRIKQogICAgICAgICAgICAgICAgICAgZm9yIHIsIGMgaW4gaW5uZXJfY2VsbHMpOgogICAgICAgICAgICAgICAgIyBDaGVjayBubyBwYXRoIGZyb20gaW5uZXIgdG8gZ3JpZCBlZGdlIHdpdGhvdXQgY3Jvc3Npbmcgb3V0ZXIKICAgICAgICAgICAgICAgICMgU2ltcGxlIGNoZWNrOiBhbGwgbmVpZ2hib3JzIG9mIGlubmVyIGNlbGxzIGFyZSBlaXRoZXIgaW5uZXIgb3Igb3V0ZXIKICAgICAgICAgICAgICAgIGlzX2NoaWxkID0gVHJ1ZQogICAgICAgICAgICAgICAgZm9yIHIsIGMgaW4gaW5uZXJfY2VsbHM6CiAgICAgICAgICAgICAgICAgICAgZm9yIGRyLCBkYyBpbiBfT1JUSDoKICAgICAgICAgICAgICAgICAgICAgICAgbnIsIG5jID0gciArIGRyLCBjICsgZGMKICAgICAgICAgICAgICAgICAgICAgICAgaWYgMCA8PSBuciA8IGhlaWdodCBhbmQgMCA8PSBuYyA8IHdpZHRoOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgaWYgY29tcF9pZFtucl1bbmNdICE9IG91dGVyX2lkIGFuZCBjb21wX2lkW25yXVtuY10gIT0gaW5uZXJfaWQ6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgaXNfY2hpbGQgPSBGYWxzZQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICAgICAgaWYgbm90IGlzX2NoaWxkOgogICAgICAgICAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICAgICAgaWYgaXNfY2hpbGQ6CiAgICAgICAgICAgICAgICAgICAgbm9kZXNbb3V0ZXJfaWRdWyJjaGlsZHJlbiJdLmFwcGVuZChpbm5lcl9pZCkKICAgIAogICAgIyBDb21wdXRlIGFkamFjZW5jeQogICAgYWRqX3NldCA9IHNldCgpCiAgICBmb3IgciBpbiByYW5nZShoZWlnaHQpOgogICAgICAgIGZvciBjIGluIHJhbmdlKHdpZHRoKToKICAgICAgICAgICAgY2lkID0gY29tcF9pZFtyXVtjXQogICAgICAgICAgICBmb3IgZHIsIGRjIGluIF9PUlRIOgogICAgICAgICAgICAgICAgbnIsIG5jID0gciArIGRyLCBjICsgZGMKICAgICAgICAgICAgICAgIGlmIDAgPD0gbnIgPCBoZWlnaHQgYW5kIDAgPD0gbmMgPCB3aWR0aDoKICAgICAgICAgICAgICAgICAgICBuaWQgPSBjb21wX2lkW25yXVtuY10KICAgICAgICAgICAgICAgICAgICBpZiBjaWQgIT0gbmlkOgogICAgICAgICAgICAgICAgICAgICAgICBwYWlyID0gKG1pbihjaWQsIG5pZCksIG1heChjaWQsIG5pZCkpCiAgICAgICAgICAgICAgICAgICAgICAgIGFkal9zZXQuYWRkKHBhaXIpCiAgICAKICAgIHJldHVybiB7CiAgICAgICAgIm5vZGVzIjogbm9kZXMsCiAgICAgICAgImFkamFjZW5jeV9saXN0Ijogc29ydGVkKGFkal9zZXQpLAogICAgfQoKCmRlZiBjb21wYWN0X3NlZ21lbnRhdGlvbl9zdW1tYXJ5KHNlZzogZGljdCwgbWF4X25vZGVzOiBpbnQgPSAyMCkgLT4gc3RyOgogICAgIiIiQ3JlYXRlIGEgY29tcGFjdCB0ZXh0IHN1bW1hcnkgb2Ygc2VnbWVudGF0aW9uIGZvciBMTE0gY29udGV4dC4iIiIKICAgIG5vZGVzID0gc2VnLmdldCgibm9kZXMiLCBbXSkKICAgIGFkaiA9IHNlZy5nZXQoImFkamFjZW5jeV9saXN0IiwgW10pCiAgICAKICAgIGxpbmVzID0gW2YiT2JqZWN0czoge2xlbihub2Rlcyl9LCBBZGphY2VuY2llczoge2xlbihhZGopfSJdCiAgICBmb3Igbm9kZSBpbiBub2Rlc1s6bWF4X25vZGVzXToKICAgICAgICBib3VuZGFyeV9zdHIgPSAiLCIuam9pbihmIntyfSx7Y30iIGZvciByLCBjIGluIG5vZGVbImJvdW5kYXJ5Il1bOjRdKQogICAgICAgIGNoaWxkcmVuX3N0ciA9IGYiLCBjaGlsZHJlbj1beycsJy5qb2luKG1hcChzdHIsIG5vZGVbJ2NoaWxkcmVuJ10pKX1dIiBpZiBub2RlWyJjaGlsZHJlbiJdIGVsc2UgIiIKICAgICAgICBsaW5lcy5hcHBlbmQoCiAgICAgICAgICAgIGYiICBvYmoje25vZGVbJ2lkJ119OiBjb2xvcj17bm9kZVsnY29sb3InXX0sIHNpemU9e25vZGVbJ3BpeGVscyddfXB4LCAiCiAgICAgICAgICAgIGYiaGFzaD17bm9kZVsnaGFzaCddWzo4XX0sIGNvcm5lcnM9W3tib3VuZGFyeV9zdHJ9XXtjaGlsZHJlbl9zdHJ9IgogICAgICAgICkKICAgIGlmIGxlbihub2RlcykgPiBtYXhfbm9kZXM6CiAgICAgICAgbGluZXMuYXBwZW5kKGYiICAuLi4gYW5kIHtsZW4obm9kZXMpIC0gbWF4X25vZGVzfSBtb3JlIG9iamVjdHMiKQogICAgcmV0dXJuICJcbiIuam9pbihsaW5lcykK"))
(agent_dir / "llm_policy.py").write_bytes(base64.b64decode("..."))
(agent_dir / "llm_agent.py").write_bytes(base64.b64decode("IiIiQVJDLUFHSS0zIEFnZW50IGFkYXB0ZXIgZm9yIHRoZSBMTE0tYmFzZWQgcG9saWN5LgoKUGx1Z3MgTExNUG9saWN5IGludG8gdGhlIEFSQy1BR0ktMy1BZ2VudHMgcmVmZXJlbmNlIGZyYW1ld29yay4KVGhlIGFnZW50IGNhbGxzIHRoZSBMTE0gdG8gZ2V0IGFjdGlvbnMsIHdpdGggYW50aS1sb29wIGRldGVjdGlvbgphbmQgYWN0aW9uIGJhdGNoaW5nIGZvciBlZmZpY2llbmN5LgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG9zCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKZnJvbSB0aHJlYWRpbmcgaW1wb3J0IFJMb2NrCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCiMgRW5zdXJlIGltcG9ydHMgd29yayBib3RoIGluIGNvbXBldGl0aW9uIGFuZCBsb2NhbGx5Cl9kaXIgPSBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkKX3BhcmVudCA9IG9zLnBhdGguZGlybmFtZShfZGlyKQpmb3IgcCBpbiAoX2RpciwgX3BhcmVudCk6CiAgICBpZiBwIG5vdCBpbiBzeXMucGF0aDoKICAgICAgICBzeXMucGF0aC5pbnNlcnQoMCwgcCkKCmZyb20gYXJjZW5naW5lIGltcG9ydCBBY3Rpb25JbnB1dCwgRnJhbWVEYXRhLCBHYW1lQWN0aW9uLCBHYW1lU3RhdGUKZnJvbSBhZ2VudHMuYWdlbnQgaW1wb3J0IEFnZW50Cgp0cnk6CiAgICBmcm9tIC5sbG1fcG9saWN5IGltcG9ydCBMTE1Qb2xpY3ksIExMTUNvbmZpZywgRnJhbWVJbmZvCiAgICBmcm9tIC5ncmlkX3V0aWxzIGltcG9ydCBmb3JtYXRfZ3JpZF9hc2NpaQpleGNlcHQgSW1wb3J0RXJyb3I6CiAgICBmcm9tIGxsbV9wb2xpY3kgaW1wb3J0IExMTVBvbGljeSwgTExNQ29uZmlnLCBGcmFtZUluZm8KICAgIGZyb20gZ3JpZF91dGlscyBpbXBvcnQgZm9ybWF0X2dyaWRfYXNjaWkKCgpkZWYgX2V4dHJhY3RfZ3JpZChmcmFtZV9kYXRhOiBGcmFtZURhdGEpIC0+IHR1cGxlW3R1cGxlW2ludCwgLi4uXSwgLi4uXToKICAgICIiIkV4dHJhY3QgdGhlIG51bWVyaWMgZ3JpZCBmcm9tIGEgRnJhbWVEYXRhIG9iamVjdC4iIiIKICAgICMgVHJ5IG11bHRpcGxlIHBhdGhzIHRvIGdldCB0aGUgZ3JpZCBkYXRhCiAgICBmb3IgYXR0cl9wYXRoIGluIFsKICAgICAgICBsYW1iZGEgZjogZi5yYXdfc3RhdGUuZGF0YSwKICAgICAgICBsYW1iZGEgZjogZi5yYXdfc3RhdGUucGxhbmVzWzBdIGlmIGYucmF3X3N0YXRlLnBsYW5lcyBlbHNlIE5vbmUsCiAgICAgICAgbGFtYmRhIGY6IGYucGxhbmVzWzBdIGlmIGhhc2F0dHIoZiwgJ3BsYW5lcycpIGFuZCBmLnBsYW5lcyBlbHNlIE5vbmUsCiAgICBdOgogICAgICAgIHRyeToKICAgICAgICAgICAgZGF0YSA9IGF0dHJfcGF0aChmcmFtZV9kYXRhKQogICAgICAgICAgICBpZiBkYXRhIGlzIE5vbmU6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBpZiBoYXNhdHRyKGRhdGEsICd0b2xpc3QnKToKICAgICAgICAgICAgICAgIHJvd3MgPSBkYXRhLnRvbGlzdCgpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICByb3dzID0gbGlzdChkYXRhKQogICAgICAgICAgICBpZiByb3dzIGFuZCByb3dzWzBdOgogICAgICAgICAgICAgICAgcmV0dXJuIHR1cGxlKHR1cGxlKGludChjKSBmb3IgYyBpbiByb3cpIGZvciByb3cgaW4gcm93cykKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBjb250aW51ZQogICAgIyBBYnNvbHV0ZSBmYWxsYmFjawogICAgcmV0dXJuICgoMCwpICogNjQsKSAqIDY0CgoKZGVmIF9mcmFtZV90b19pbmZvKGZyYW1lX2RhdGE6IEZyYW1lRGF0YSkgLT4gRnJhbWVJbmZvOgogICAgIiIiQ29udmVydCBGcmFtZURhdGEgdG8gb3VyIEZyYW1lSW5mby4iIiIKICAgIGdyaWQgPSBfZXh0cmFjdF9ncmlkKGZyYW1lX2RhdGEpCiAgICAKICAgICMgU3RhdGUgbmFtZQogICAgdHJ5OgogICAgICAgIHN0YXRlID0gZnJhbWVfZGF0YS5zdGF0ZQogICAgICAgIHN0YXRlX25hbWUgPSBzdGF0ZS5uYW1lIGlmIGhhc2F0dHIoc3RhdGUsICduYW1lJykgZWxzZSBzdHIoc3RhdGUpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHN0YXRlX25hbWUgPSAiUExBWUlORyIKICAgIAogICAgIyBMZXZlbHMgY29tcGxldGVkCiAgICB0cnk6CiAgICAgICAgbGV2ZWxzX2NvbXBsZXRlZCA9IGludChmcmFtZV9kYXRhLmxldmVsc19jb21wbGV0ZWQpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIGxldmVsc19jb21wbGV0ZWQgPSAwCiAgICAKICAgICMgQXZhaWxhYmxlIGFjdGlvbnMKICAgIGF2YWlsYWJsZSA9IFtdCiAgICB0cnk6CiAgICAgICAgZm9yIGFjdF9pZCBpbiBmcmFtZV9kYXRhLmF2YWlsYWJsZV9hY3Rpb25zOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBhY3QgPSBHYW1lQWN0aW9uLmZyb21faWQoaW50KGFjdF9pZCkpCiAgICAgICAgICAgICAgICBpZiBhY3QubmFtZSAhPSAiUkVTRVQiOgogICAgICAgICAgICAgICAgICAgIGF2YWlsYWJsZS5hcHBlbmQoYWN0Lm5hbWUpCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICAgICBwYXNzCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIGF2YWlsYWJsZSA9IFsiQUNUSU9OMSIsICJBQ1RJT04yIiwgIkFDVElPTjMiLCAiQUNUSU9ONCIsICJBQ1RJT041IiwgIkFDVElPTjYiLCAiQUNUSU9ONyJdCiAgICAKICAgICMgU3RlcCBudW1iZXIKICAgIHRyeToKICAgICAgICBzdGVwID0gaW50KGZyYW1lX2RhdGEuc3RlcCkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgc3RlcCA9IDAKICAgIAogICAgcmV0dXJuIEZyYW1lSW5mbygKICAgICAgICBncmlkPWdyaWQsCiAgICAgICAgc3RlcD1zdGVwLAogICAgICAgIGxldmVsPWxldmVsc19jb21wbGV0ZWQgKyAxLAogICAgICAgIHN0YXRlX25hbWU9c3RhdGVfbmFtZSwKICAgICAgICBsZXZlbHNfY29tcGxldGVkPWxldmVsc19jb21wbGV0ZWQsCiAgICAgICAgYXZhaWxhYmxlX2FjdGlvbnM9YXZhaWxhYmxlLAogICAgKQoKCmNsYXNzIExMTUFnZW50KEFnZW50KToKICAgICIiIkxMTS1wb3dlcmVkIGFnZW50IGZvciBBUkMtQUdJLTMuIiIiCiAgICAKICAgIF9BQ1RJT05fTE9DSyA9IFJMb2NrKCkKICAgIAogICAgQUNUSU9OX0JVREdFVCA9IDQwMAogICAgTUFYX0FDVElPTlMgPSBBQ1RJT05fQlVER0VUCiAgICAKICAgICMgU29mdCB0aW1lIGxpbWl0IHBlciBnYW1lIChzZWNvbmRzKQogICAgIyAxMTAgZ2FtZXMgw5cgNCBtaW4gPSA0NDAgbWluIOKJiCA3LjMgaG91cnMg4oCUIGZpdHMgaW4gOWggd2l0aCBtYXJnaW4KICAgIEdBTUVfVElNRV9MSU1JVCA9IDI0MAogICAgCiAgICBkZWYgX19pbml0X18oc2VsZiwgKmFyZ3M6IEFueSwgKiprd2FyZ3M6IEFueSkgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCphcmdzLCAqKmt3YXJncykKICAgICAgICBzZWxmLmNvbmZpZyA9IExMTUNvbmZpZy5mcm9tX2VudigpCiAgICAgICAgc2VsZi5wb2xpY3kgPSBMTE1Qb2xpY3koc2VsZi5jb25maWcpCiAgICAgICAgc2VsZi5faG9sZHNfYWN0aW9uX2xvY2sgPSBGYWxzZQogICAgICAgIHNlbGYuX2dhbWVfc3RhcnRfdGltZSA9IHRpbWUudGltZSgpCiAgICAgICAgc2VsZi5fYWN0aW9uc19pbl9nYW1lID0gMAogICAgICAgIHNlbGYuX2ZpcnN0X2FjdGlvbiA9IFRydWUKICAgIAogICAgQHByb3BlcnR5CiAgICBkZWYgbmFtZShzZWxmKSAtPiBzdHI6CiAgICAgICAgcmV0dXJuIGYie3N1cGVyKCkubmFtZX0ubGxtLXYyIgogICAgCiAgICBkZWYgaXNfZG9uZShzZWxmLCBmcmFtZXM6IGxpc3RbRnJhbWVEYXRhXSwgbGF0ZXN0X2ZyYW1lOiBGcmFtZURhdGEpIC0+IGJvb2w6CiAgICAgICAgIiIiU3RvcCBjb25kaXRpb25zOiB3aW4sIGJ1ZGdldCwgb3IgdGltZSBsaW1pdC4iIiIKICAgICAgICBpZiBzZWxmLl9hY3Rpb25zX2luX2dhbWUgPj0gc2VsZi5BQ1RJT05fQlVER0VUOgogICAgICAgICAgICByZXR1cm4gVHJ1ZQogICAgICAgIHRyeToKICAgICAgICAgICAgaWYgbGF0ZXN0X2ZyYW1lLnN0YXRlIGlzIEdhbWVTdGF0ZS5XSU46CiAgICAgICAgICAgICAgICByZXR1cm4gVHJ1ZQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHBhc3MKICAgICAgICBpZiB0aW1lLnRpbWUoKSAtIHNlbGYuX2dhbWVfc3RhcnRfdGltZSA+IHNlbGYuR0FNRV9USU1FX0xJTUlUOgogICAgICAgICAgICByZXR1cm4gVHJ1ZQogICAgICAgIHJldHVybiBGYWxzZQogICAgCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgX3RvX2dhbWVfYWN0aW9uKHByb3Bvc2FsOiBkaWN0KSAtPiBHYW1lQWN0aW9uOgogICAgICAgICIiIkNvbnZlcnQgcG9saWN5IG91dHB1dCB0byBHYW1lQWN0aW9uIGVudW0uIiIiCiAgICAgICAgYWN0aW9uX25hbWUgPSBwcm9wb3NhbC5nZXQoIm5hbWUiLCAiUkVTRVQiKQogICAgICAgIAogICAgICAgIHRyeToKICAgICAgICAgICAgYWN0aW9uID0gR2FtZUFjdGlvblthY3Rpb25fbmFtZV0KICAgICAgICBleGNlcHQgS2V5RXJyb3I6CiAgICAgICAgICAgIGFjdGlvbiA9IEdhbWVBY3Rpb24uUkVTRVQKICAgICAgICAKICAgICAgICBpZiBhY3Rpb24uaXNfY29tcGxleCgpOgogICAgICAgICAgICB4ID0gcHJvcG9zYWwuZ2V0KCJ4IiwgMCkgb3IgMAogICAgICAgICAgICB5ID0gcHJvcG9zYWwuZ2V0KCJ5IiwgMCkgb3IgMAogICAgICAgICAgICBhY3Rpb24uc2V0X2RhdGEoeyJ4IjogbWF4KDAsIG1pbig2MywgaW50KHgpKSksICJ5IjogbWF4KDAsIG1pbig2MywgaW50KHkpKSl9KQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGFjdGlvbi5zZXRfZGF0YSh7fSkKICAgICAgICAKICAgICAgICBhY3Rpb24ucmVhc29uaW5nID0gcHJvcG9zYWwuZ2V0KCJyZWFzb25pbmciLCB7fSkKICAgICAgICByZXR1cm4gYWN0aW9uCiAgICAKICAgIGRlZiBjaG9vc2VfYWN0aW9uKAogICAgICAgIHNlbGYsIGZyYW1lczogbGlzdFtGcmFtZURhdGFdLCBsYXRlc3RfZnJhbWU6IEZyYW1lRGF0YQogICAgKSAtPiBHYW1lQWN0aW9uOgogICAgICAgICIiIk1haW4gZW50cnkgcG9pbnQgY2FsbGVkIGJ5IHRoZSBmcmFtZXdvcmsgZWFjaCB0dXJuLiIiIgogICAgICAgIHNlbGYuX0FDVElPTl9MT0NLLmFjcXVpcmUoKQogICAgICAgIHNlbGYuX2hvbGRzX2FjdGlvbl9sb2NrID0gVHJ1ZQogICAgICAgIAogICAgICAgIHRyeToKICAgICAgICAgICAgZnJhbWVfaW5mbyA9IF9mcmFtZV90b19pbmZvKGxhdGVzdF9mcmFtZSkKICAgICAgICAgICAgCiAgICAgICAgICAgICMgSW5pdGlhbGl6ZSBvbiBmaXJzdCBjYWxsCiAgICAgICAgICAgIGlmIHNlbGYuX2ZpcnN0X2FjdGlvbjoKICAgICAgICAgICAgICAgIHNlbGYuX2ZpcnN0X2FjdGlvbiA9IEZhbHNlCiAgICAgICAgICAgICAgICBzZWxmLl9nYW1lX3N0YXJ0X3RpbWUgPSB0aW1lLnRpbWUoKQogICAgICAgICAgICAgICAgc2VsZi5wb2xpY3kucmVzZXRfZ2FtZSgpCiAgICAgICAgICAgIAogICAgICAgICAgICAjIEF1dG8tcmVzZXQgb24gR0FNRV9PVkVSCiAgICAgICAgICAgIGlmIGZyYW1lX2luZm8uc3RhdGVfbmFtZSA9PSAiR0FNRV9PVkVSIjoKICAgICAgICAgICAgICAgIGFjdGlvbiA9IEdhbWVBY3Rpb24uUkVTRVQKICAgICAgICAgICAgICAgIGFjdGlvbi5zZXRfZGF0YSh7fSkKICAgICAgICAgICAgICAgIGFjdGlvbi5yZWFzb25pbmcgPSB7ImF1dG9fcmVzZXQiOiBUcnVlLCAiZ2FtZV9vdmVyIjogVHJ1ZX0KICAgICAgICAgICAgICAgIHJldHVybiBhY3Rpb24KICAgICAgICAgICAgCiAgICAgICAgICAgICMgQ2FsbCB0aGUgTExNIHBvbGljeQogICAgICAgICAgICBwcm9wb3NhbCA9IHNlbGYucG9saWN5LmNob29zZV9hY3Rpb24oZnJhbWVfaW5mbykKICAgICAgICAgICAgc2VsZi5fYWN0aW9uc19pbl9nYW1lICs9IDEKICAgICAgICAgICAgCiAgICAgICAgICAgIHJldHVybiBzZWxmLl90b19nYW1lX2FjdGlvbihwcm9wb3NhbCkKICAgICAgICAgICAgCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgICAgICBpbXBvcnQgdHJhY2ViYWNrCiAgICAgICAgICAgIHByaW50KGYiW0xMTUFnZW50XSBFcnJvcjoge2V9IikKICAgICAgICAgICAgdHJhY2ViYWNrLnByaW50X2V4YygpCiAgICAgICAgICAgIGFjdGlvbiA9IEdhbWVBY3Rpb24uUkVTRVQKICAgICAgICAgICAgYWN0aW9uLnNldF9kYXRhKHt9KQogICAgICAgICAgICBhY3Rpb24ucmVhc29uaW5nID0geyJlcnJvciI6IHN0cihlKX0KICAgICAgICAgICAgcmV0dXJuIGFjdGlvbgogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIGlmIHNlbGYuX2hvbGRzX2FjdGlvbl9sb2NrOgogICAgICAgICAgICAgICAgc2VsZi5faG9sZHNfYWN0aW9uX2xvY2sgPSBGYWxzZQogICAgICAgICAgICAgICAgc2VsZi5fQUNUSU9OX0xPQ0sucmVsZWFzZSgpCiAgICAKICAgIGRlZiB0YWtlX2FjdGlvbihzZWxmLCBhY3Rpb246IEdhbWVBY3Rpb24pOiAgIyB0eXBlOiBpZ25vcmVbb3ZlcnJpZGVdCiAgICAgICAgIiIiRXhlY3V0ZSBhY3Rpb24gYW5kIHJlY29yZCB0aGUgcmVzdWx0LiIiIgogICAgICAgIHRyeToKICAgICAgICAgICAgZnJhbWUgPSBzdXBlcigpLnRha2VfYWN0aW9uKGFjdGlvbikKICAgICAgICAgICAgaWYgZnJhbWUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICBmcmFtZS5hY3Rpb25faW5wdXQgPSBBY3Rpb25JbnB1dCgKICAgICAgICAgICAgICAgICAgICBpZD1hY3Rpb24sCiAgICAgICAgICAgICAgICAgICAgZGF0YT1hY3Rpb24uYWN0aW9uX2RhdGEubW9kZWxfZHVtcCgpLAogICAgICAgICAgICAgICAgICAgIHJlYXNvbmluZz1nZXRhdHRyKGFjdGlvbiwgInJlYXNvbmluZyIsIE5vbmUpLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICByZXR1cm4gZnJhbWUKICAgICAgICBmaW5hbGx5OgogICAgICAgICAgICBpZiBnZXRhdHRyKHNlbGYsICJfaG9sZHNfYWN0aW9uX2xvY2siLCBGYWxzZSk6CiAgICAgICAgICAgICAgICBzZWxmLl9ob2xkc19hY3Rpb25fbG9jayA9IEZhbHNlCiAgICAgICAgICAgICAgICBzZWxmLl9BQ1RJT05fTE9DSy5yZWxlYXNlKCkKICAgIAogICAgZGVmIGZpbmFsaXplX2RpYWdub3N0aWNzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgIiIiTG9nIGZpbmFsIHN0YXRzIGZvciB0aGlzIGdhbWUuIiIiCiAgICAgICAgZWxhcHNlZCA9IHRpbWUudGltZSgpIC0gc2VsZi5fZ2FtZV9zdGFydF90aW1lCiAgICAgICAgcHJpbnQoZiJbTExNQWdlbnRdIEdhbWUgZG9uZTogIgogICAgICAgICAgICAgIGYie3NlbGYuX2FjdGlvbnNfaW5fZ2FtZX0gYWN0aW9ucyBpbiB7ZWxhcHNlZDouMGZ9cywgIgogICAgICAgICAgICAgIGYie3NlbGYucG9saWN5LnRvdGFsX3Rva2Vuc30gdG9rZW5zICh7c2VsZi5wb2xpY3kudG90YWxfY2FsbHN9IGNhbGxzLCAiCiAgICAgICAgICAgICAgZiJ7c2VsZi5wb2xpY3kuY2xpZW50LnRvdGFsX3RpbWU6LjBmfXMgTExNIHRpbWUpLCAiCiAgICAgICAgICAgICAgZiJ7bGVuKHNlbGYucG9saWN5LnRyYWNrZXIudW5pcXVlX3N0YXRlc19zZWVuKX0gdW5pcXVlIHN0YXRlcywgIgogICAgICAgICAgICAgIGYie3NlbGYucG9saWN5LnRyYWNrZXIuZ2FtZV9vdmVyX2NvdW50fSBkZWF0aHMiKQo="))

# Set environment for the agent
os.environ["LLM_BASE_URL"] = VLLM_BASE_URL
os.environ["LLM_MODEL_ID"] = SERVED_MODEL
os.environ["LLM_API_KEY"] = "EMPTY"
os.environ["LLM_CONTEXT_WINDOW"] = "32768"
os.environ["LLM_MAX_OUTPUT"] = "2048"
os.environ["LLM_TEMPERATURE"] = "0.6"
os.environ["LLM_TIMEOUT"] = "120"
os.environ["LLM_MULTIMODAL"] = "0"  # start without images to save tokens

print(f"Agent deployed in {agent_dir}")
print(f"LLM config: {SERVED_MODEL} @ {VLLM_BASE_URL}")


In [ ]:
import shutil
from pathlib import Path

if TRUE_SUBMISSION:
    # === COMPETITION RUN ===
    # Copy the ARC-AGI-3-Agents framework
    framework_src = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents")
    framework = WORKING / "ARC-AGI-3-Agents"
    if framework.exists():
        shutil.rmtree(framework)
    shutil.copytree(framework_src, framework)
    
    # Copy our agent into the templates directory
    target = framework / "agents" / "templates"
    target.mkdir(parents=True, exist_ok=True)
    for f in ("__init__.py", "grid_utils.py", "llm_policy.py", "llm_agent.py"):
        shutil.copy2(Path("/tmp/arc_llm_agent") / f, target / f)
    
    # Register our agent
    init_content = """
\"\"\"Agent registry for ARC-AGI-3 LLM agent.\"\"\"
from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .recorder import Recorder
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent
from .templates.llm_agent import LLMAgent

load_dotenv()
AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
    "llmagent": LLMAgent,
}
for recording in Recorder.list():
    AVAILABLE_AGENTS[recording] = Playback
"""
    (framework / "agents" / "__init__.py").write_text(init_content)
    
    # Write .env
    (framework / ".env").write_text("\n".join([
        "SCHEME=http",
        "HOST=gateway",
        "PORT=8001",
        "ARC_API_KEY=competition-gateway",
        "ARC_BASE_URL=http://gateway:8001/",
        "OPERATION_MODE=online",
        "ENVIRONMENTS_DIR=",
        f"RECORDINGS_DIR={WORKING}/recordings",
        f"LLM_BASE_URL={VLLM_BASE_URL}",
        f"LLM_MODEL_ID={SERVED_MODEL}",
        "LLM_API_KEY=EMPTY",
        "LLM_CONTEXT_WINDOW=32768",
        "LLM_MAX_OUTPUT=2048",
        "LLM_TEMPERATURE=0.6",
        "LLM_TIMEOUT=120",
        "LLM_MULTIMODAL=0",
    ]) + "\n")
    
    # Wait for gateway
    import urllib.request
    print("Waiting for competition gateway...")
    deadline = time.time() + 600
    while time.time() < deadline:
        try:
            with urllib.request.urlopen("http://gateway:8001/api/games", timeout=10) as r:
                if r.status < 500:
                    break
        except Exception:
            pass
        time.sleep(5)
    
    # Run the agent
    run_env = os.environ.copy()
    run_env["MPLBACKEND"] = "Agg"
    run_env["LLM_BASE_URL"] = VLLM_BASE_URL
    run_env["LLM_MODEL_ID"] = SERVED_MODEL
    run_env["PYTHONPATH"] = str(SITE_PACKAGES) + os.pathsep + run_env.get("PYTHONPATH", "")
    
    # Choose agent: LLM if vLLM started successfully, otherwise fallback to heuristic
    if VLLM_OK:
        agent_name = "llmagent"
        print(f"Starting LLM agent run ({agent_name})...")
    else:
        agent_name = "myagent"
        print("WARNING: vLLM not available, falling back to heuristic agent (myagent)")
    
    result = subprocess.run(
        [sys.executable, "main.py", "--agent", agent_name],
        cwd=framework,
        env=run_env,
        capture_output=True,
        text=True,
        timeout=32000,  # ~9 hours
    )
    print("STDOUT:", result.stdout[-3000:] if result.stdout else "(empty)")
    print("STDERR:", result.stderr[-3000:] if result.stderr else "(empty)")
    print(f"Return code: {result.returncode}")
    
else:
    # === SAVE PHASE (non-competition) ===
    import pandas as pd
    placeholder = pd.DataFrame(
        [["placeholder_0", "placeholder", True, 0.0]],
        columns=["row_id", "game_id", "end_of_game", "score"],
    )
    placeholder.to_parquet(WORKING / "submission.parquet", index=False)
    print("Wrote save-phase placeholder submission.parquet")


In [ ]:
# Print diagnostics
elapsed = time.time() - NOTEBOOK_START
print(f"Notebook elapsed: {elapsed:.0f}s ({elapsed/60:.1f} min)")

# Show vLLM server log tail
if VLLM_LOG.exists():
    log = VLLM_LOG.read_text()
    print(f"\nvLLM log ({len(log)} chars), last 500:")
    print(log[-500:])

# Show any recordings
recordings = WORKING / "recordings"
if recordings.exists():
    files = list(recordings.iterdir())
    print(f"\nRecordings: {len(files)} files")
    for f in files[:5]:
        print(f"  {f.name} ({f.stat().st_size} bytes)")
